[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AltamarMx/anomalias-2026-2/blob/main/notebooks/013_componentes_teoria.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
rng = np.random.default_rng(42)
import ipywidgets as widgets
from ipywidgets import interact, interactive_output
from IPython.display import display, Markdown

# Semana 7A — Componentes de Series Temporales (Teoría)

**Objetivo:** Entender que una serie temporal se puede descomponer en
componentes interpretables (tendencia, estacionalidad, ruido) y cuándo
usar un modelo aditivo vs multiplicativo.

**Temas:**

1. Anatomía de una serie temporal
2. Modelo aditivo vs multiplicativo
3. Descomposición manual paso a paso
4. ¿Cuándo usar cada modelo?

---
## 1. Anatomía de una serie temporal

Toda serie temporal se puede pensar como la superposición de:

| Componente | Descripción | Escala típica |
|:---|:---|:---|
| **Tendencia (T)** | Cambio lento y sostenido del nivel | Años |
| **Estacionalidad (S)** | Patrón que se repite con período fijo | Horas, días, meses |
| **Ruido (R)** | Variación aleatoria sin estructura | Instantánea |

En datos meteorológicos:
- Tendencia = calentamiento gradual (o cambios de instrumento)
- Estacionalidad = ciclo diurno (24 h) + ciclo anual (365 d)
- Ruido = variabilidad del clima hora a hora

### Construye tu propia serie temporal

In [ ]:
amp_tend_slider = widgets.FloatSlider(min=0.0, max=0.05, step=0.005, value=0.01, description="Pendiente de tendencia")
amp_estac_slider = widgets.FloatSlider(min=0.0, max=10.0, step=0.5, value=5.0, description="Amplitud de estacionalidad")
amp_ruido_slider = widgets.FloatSlider(min=0.0, max=5.0, step=0.25, value=1.0, description="Amplitud del ruido (σ)",
)

def _update(amp_tend_slider, amp_estac_slider, amp_ruido_slider):
    _n = 730  # ~2 años en días
    _t = np.arange(_n)

    _tendencia = amp_tend_slider * _t
    _estacionalidad = amp_estac_slider * np.sin(2 * np.pi * _t / 365)
    _ruido = amp_ruido_slider * rng.normal(0, 1, _n)
    _serie = _tendencia + _estacionalidad + _ruido

    fig1, axes1 = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

    axes1[0].plot(_t, _serie, "steelblue", lw=0.8)
    axes1[0].set_ylabel("y(t)")
    axes1[0].set_title("Serie observada = Tendencia + Estacionalidad + Ruido")

    axes1[1].plot(_t, _tendencia, "crimson", lw=2)
    axes1[1].set_ylabel("Tendencia")
    axes1[1].set_title(f"Tendencia (pendiente = {amp_tend_slider})")

    axes1[2].plot(_t, _estacionalidad, "seagreen", lw=1.5)
    axes1[2].set_ylabel("Estacionalidad")
    axes1[2].set_title(f"Estacionalidad (amplitud = {amp_estac_slider})")

    axes1[3].plot(_t, _ruido, "gray", lw=0.5, alpha=0.7)
    axes1[3].set_ylabel("Ruido")
    axes1[3].set_xlabel("Día")
    axes1[3].set_title(f"Ruido (σ = {amp_ruido_slider})")

    plt.tight_layout()
    plt.show()

out = widgets.interactive_output(_update, {'amp_tend_slider': amp_tend_slider, 'amp_estac_slider': amp_estac_slider, 'amp_ruido_slider': amp_ruido_slider})
display(widgets.VBox([amp_tend_slider, amp_estac_slider, amp_ruido_slider, out]))

> **Experimenta:**
> - Pon ruido = 0: la serie es perfectamente predecible.
> - Sube el ruido a 5: la estacionalidad se esconde bajo el ruido.
> - Pon tendencia = 0.05: después de 2 años, el nivel subió ~36 unidades.
> - Pon estacionalidad = 0: la serie es solo tendencia + ruido (sin patrón cíclico).

---
## 2. Modelo aditivo vs multiplicativo

| Modelo | Ecuación | Cuándo usarlo |
|:---|:---|:---|
| **Aditivo** | $y = T + S + R$ | La amplitud de la estacionalidad es constante |
| **Multiplicativo** | $y = T \times S \times R$ | La amplitud crece (o decrece) con el nivel |

**Ejemplo meteorológico:**
- **Temperatura:** oscila ±5°C en verano y ±5°C en invierno → **aditivo**
- **Irradiancia solar:** oscila ±200 W/m² en verano pero ±50 W/m² en invierno
  (la amplitud es proporcional al nivel) → **multiplicativo**

In [ ]:
_n = 730
_t = np.arange(_n)

# Tendencia creciente
_T = 10 + 0.02 * _t

# Estacionalidad normalizada (oscila entre 0.8 y 1.2)
_S_mult = 1 + 0.3 * np.sin(2 * np.pi * _t / 365)
# Estacionalidad absoluta (oscila ±3)
_S_add = 3 * np.sin(2 * np.pi * _t / 365)

_R_add = rng.normal(0, 1, _n)
_R_mult = 1 + rng.normal(0, 0.05, _n)

_y_add = _T + _S_add + _R_add
_y_mult = _T * _S_mult * _R_mult

fig2, (ax2a, ax2b) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax2a.plot(_t, _y_add, "steelblue", lw=0.8)
ax2a.set_ylabel("y(t)")
ax2a.set_title("Modelo ADITIVO: y = T + S + R  (amplitud constante)")

ax2b.plot(_t, _y_mult, "darkorange", lw=0.8)
ax2b.set_ylabel("y(t)")
ax2b.set_xlabel("Día")
ax2b.set_title("Modelo MULTIPLICATIVO: y = T × S × R  (amplitud crece con el nivel)")

plt.tight_layout()
plt.show()

> **Observa la diferencia clave:**
> - En el **aditivo**, la "banda" de oscilación tiene ancho constante a lo largo del tiempo.
> - En el **multiplicativo**, la banda se ensancha conforme la tendencia crece.
>
> Si aplicas descomposición aditiva a datos multiplicativos, los residuos
> tendrán varianza creciente (heterocedasticidad) — señal de modelo inadecuado.

---
## 3. Descomposición manual paso a paso

Antes de usar funciones automáticas, vamos a descomponer "a mano"
una serie aditiva sintética. Esto desmitifica lo que hace
`seasonal_decompose` internamente.

**Pasos:**

1. **Estimar la tendencia** con media móvil centrada (ventana = período)
2. **Restar la tendencia** → residuo que contiene estacionalidad + ruido
3. **Estimar la estacionalidad** promediando por posición en el ciclo
4. **Restar la estacionalidad** → residuo puro

In [ ]:
# Generar serie sintética con período = 50 (para visualización clara)
_periodo = 50
_n = 500
_t = np.arange(_n)

_T_real = 20 + 0.01 * _t
_S_real = 4 * np.sin(2 * np.pi * _t / _periodo)
_R_real = rng.normal(0, 1, _n)
_y = _T_real + _S_real + _R_real

# Paso 1: Media móvil centrada
_kernel = np.ones(_periodo) / _periodo
_T_est = np.convolve(_y, _kernel, mode="same")
# Corregir bordes (la convolución no está bien definida ahí)
_margen = _periodo // 2
_T_est[:_margen] = np.nan
_T_est[-_margen:] = np.nan

# Paso 2: Restar tendencia
_detrended = _y - _T_est

# Paso 3: Promediar por posición en el ciclo
_S_est = np.full(_n, np.nan)
for pos in range(_periodo):
    _indices = np.arange(pos, _n, _periodo)
    _vals = _detrended[_indices]
    _S_est[_indices] = np.nanmean(_vals)

# Paso 4: Residuo
_R_est = _y - _T_est - _S_est

fig3, axes3 = plt.subplots(5, 1, figsize=(12, 12), sharex=True)

axes3[0].plot(_t, _y, "steelblue", lw=0.8)
axes3[0].set_ylabel("y(t)")
axes3[0].set_title("Serie observada")

axes3[1].plot(_t, _T_real, "crimson", lw=1, ls="--", label="Real")
axes3[1].plot(_t, _T_est, "crimson", lw=2, label="Estimada (media móvil)")
axes3[1].set_ylabel("Tendencia")
axes3[1].set_title("Paso 1: Estimar tendencia con media móvil")
axes3[1].legend()

axes3[2].plot(_t, _detrended, "gray", lw=0.8)
axes3[2].set_ylabel("y − T̂")
axes3[2].set_title("Paso 2: Restar tendencia")

axes3[3].plot(_t, _S_real, "seagreen", lw=1, ls="--", label="Real")
axes3[3].plot(_t, _S_est, "seagreen", lw=2, label="Estimada (promedio por posición)")
axes3[3].set_ylabel("Estacionalidad")
axes3[3].set_title("Paso 3: Estimar estacionalidad")
axes3[3].legend()

axes3[4].plot(_t, _R_real, "gray", lw=0.5, alpha=0.5, label="Real")
axes3[4].plot(_t, _R_est, "purple", lw=0.8, alpha=0.7, label="Estimado")
axes3[4].set_ylabel("Residuo")
axes3[4].set_xlabel("Tiempo")
axes3[4].set_title("Paso 4: Residuo = y − T̂ − Ŝ")
axes3[4].legend()

plt.tight_layout()
plt.show()

> **Nota:** La tendencia estimada tiene NaN en los bordes porque la
> media móvil centrada necesita `período/2` datos a cada lado.
> Esto es una limitación inherente — `seasonal_decompose` de
> statsmodels tiene el mismo problema.

---
## 4. ¿Cuándo usar cada modelo? — Diagnóstico visual

**Regla práctica:** Divide la serie en ventanas de igual tamaño.
Para cada ventana, calcula la media y la desviación estándar.

- Si la DE **no cambia** con la media → **aditivo**
- Si la DE **crece** con la media → **multiplicativo**

In [ ]:
_n = 730
_t = np.arange(_n)
_T = 10 + 0.03 * _t

# Serie aditiva
_S_a = 3 * np.sin(2 * np.pi * _t / 365)
_y_a = _T + _S_a + rng.normal(0, 1, _n)

# Serie multiplicativa
_S_m = 1 + 0.3 * np.sin(2 * np.pi * _t / 365)
_y_m = _T * _S_m * (1 + rng.normal(0, 0.05, _n))

_ventana = 60

fig4, axes4 = plt.subplots(1, 2, figsize=(12, 5))

for ax, y, titulo, color in [
    (axes4[0], _y_a, "Aditivo", "steelblue"),
    (axes4[1], _y_m, "Multiplicativo", "darkorange"),
]:
    _medias = []
    _des = []
    for start in range(0, _n - _ventana, _ventana):
        _bloque = y[start : start + _ventana]
        _medias.append(_bloque.mean())
        _des.append(_bloque.std())
    ax.scatter(_medias, _des, c=color, s=40, alpha=0.7, edgecolors="white")

    # Línea de regresión
    _coef = np.polyfit(_medias, _des, 1)
    _x_fit = np.array([min(_medias), max(_medias)])
    ax.plot(_x_fit, np.polyval(_coef, _x_fit), "k--", lw=1.5,
            label=f"pendiente = {_coef[0]:.3f}")

    ax.set_xlabel("Media de la ventana")
    ax.set_ylabel("DE de la ventana")
    ax.set_title(f"Modelo {titulo}")
    ax.legend()

plt.suptitle(
    f"Diagnóstico: DE vs Media por ventanas de {_ventana} días",
    fontsize=13, y=1.02,
)
plt.tight_layout()
plt.show()

> **Resultado:**
> - **Aditivo:** los puntos se distribuyen como una nube horizontal
>   (la DE no depende de la media). Pendiente ≈ 0.
> - **Multiplicativo:** los puntos forman una tendencia creciente
>   (la DE crece con la media). Pendiente > 0.
>
> Este diagnóstico se aplica igual a datos reales — lo haremos en la
> sesión de laboratorio con `tdb` y `ghi`.

---
## Resumen

| Concepto | Idea clave |
|:---|:---|
| **Componentes** | Toda serie = Tendencia + Estacionalidad + Ruido |
| **Aditivo** | $y = T + S + R$ — amplitud constante |
| **Multiplicativo** | $y = T \times S \times R$ — amplitud proporcional al nivel |
| **Descomposición manual** | Media móvil → tendencia; promediar por posición → estacionalidad; residuo = lo que queda |
| **Diagnóstico** | Graficar DE vs Media por ventanas para decidir aditivo/multiplicativo |

**Próxima sesión (7B):** Descomponer `tdb` y `ghi` de ClimaLab
y verificar qué modelo conviene para cada variable.